In [4]:
import sys

import matplotlib
import numpy as np
import pandas as pd
import sklearn

print("Student ID: CIT-24-01-0251")
print("Python executable:", sys.executable)
print("Python version:", sys.version)
print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)
print("Scikit-learn version:", sklearn.__version__)
print("Matplotlib version:", matplotlib.__version__)
print("\nEnvironment setup completed successfully.")

Student ID: CIT-24-01-0251
Python executable: f:\BSc (Hons) in Cyber Security\Projects\NLP_Group_02\.venv\Scripts\python.exe
Python version: 3.12.5 (tags/v3.12.5:ff3bc82, Aug  6 2024, 20:45:27) [MSC v.1940 64 bit (AMD64)]
Pandas version: 3.0.5
NumPy version: 2.5.1
Scikit-learn version: 1.9.0
Matplotlib version: 3.11.1

Environment setup completed successfully.


In [5]:
from pathlib import Path

# Locate the repository root.
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "diversevul_20230702.csv"
)

print("Current working directory:", Path.cwd())
print("Project root:", PROJECT_ROOT)
print("Dataset path:", DATA_PATH)
print("Dataset exists:", DATA_PATH.exists())

if DATA_PATH.exists():
    size_mb = DATA_PATH.stat().st_size / (1024 * 1024)
    print(f"Dataset size: {size_mb:.2f} MB")
else:
    print("Dataset was not found.")

Current working directory: f:\BSc (Hons) in Cyber Security\Projects\NLP_Group_02\notebooks
Project root: f:\BSc (Hons) in Cyber Security\Projects\NLP_Group_02
Dataset path: f:\BSc (Hons) in Cyber Security\Projects\NLP_Group_02\data\raw\diversevul_20230702.csv
Dataset exists: True
Dataset size: 626.98 MB


In [6]:
# Read only five records to inspect the dataset structure safely.
sample_df = pd.read_csv(
    DATA_PATH,
    nrows=5,
    encoding="utf-8"
)

print("Detected columns:")
print(sample_df.columns.tolist())

print("\nSample shape:", sample_df.shape)
print("\nData types:")
print(sample_df.dtypes)

Detected columns:
['func', 'target', 'cwe', 'project', 'commit_id', 'hash', 'size', 'message']

Sample shape: (5, 8)

Data types:
func            str
target        int64
cwe             str
project         str
commit_id       str
hash         object
size          int64
message         str
dtype: object


In [7]:
# Create a readable preview without displaying extremely long source-code fields.
sample_preview = sample_df.copy()

sample_preview["func_preview"] = (
    sample_preview["func"]
    .astype(str)
    .str.replace("\n", " ", regex=False)
    .str.slice(0, 200)
)

display(
    sample_preview[
        ["func_preview", "target", "cwe", "project", "size"]
    ]
)

,func_preview,target,cwe,project,size
0,int _gnutls_ciphertext2compressed(gnutls_sessi...,1,NaN,gnutls,157
1,static char *make_filename_safe(const char *fi...,1,CWE-264,php-src,22
2,"unpack_Z_stream(int fd_in, int fd_out) { \tIF_...",1,NaN,busybox,232
3,"static void cirrus_do_copy(CirrusVGAState *s, ...",1,CWE-787,qemu,66
4,"glue(cirrus_bitblt_rop_fwd_, ROP_NAME)(CirrusV...",1,CWE-787,qemu,18


In [8]:
# Scan the large dataset in chunks without loading all records into memory.

from collections import Counter

CHUNK_SIZE = 10_000

total_rows = 0
label_counts = Counter()
missing_func = 0
missing_target = 0

for chunk in pd.read_csv(
    DATA_PATH,
    usecols=["func", "target"],
    chunksize=CHUNK_SIZE,
    encoding="utf-8"
):
    total_rows += len(chunk)

    label_counts.update(
        chunk["target"].dropna().astype(int).tolist()
    )

    missing_func += chunk["func"].isna().sum()
    missing_target += chunk["target"].isna().sum()

print("Dataset analysis completed.")
print("Total records:", total_rows)
print("Label counts:", dict(label_counts))
print("Missing source-code records:", missing_func)
print("Missing target labels:", missing_target)

for label, count in sorted(label_counts.items()):
    percentage = (count / total_rows) * 100
    label_name = "Non-vulnerable" if label == 0 else "Vulnerable"

    print(
        f"Class {label} ({label_name}): "
        f"{count:,} records ({percentage:.2f}%)"
    )

Dataset analysis completed.
Total records: 320782
Label counts: {1: 18945, 0: 301837}
Missing source-code records: 1
Missing target labels: 0
Class 0 (Non-vulnerable): 301,837 records (94.09%)
Class 1 (Vulnerable): 18,945 records (5.91%)


In [9]:
# Create a reproducible 10,000-record stratified working dataset
# without loading the complete dataset into memory.

import math
import random

SAMPLE_SIZE = 10_000
RANDOM_SEED = 42
CHUNK_SIZE = 10_000

# Counts identified during the complete dataset scan.
class_counts = {
    0: 301_837,  # Non-vulnerable
    1: 18_945,   # Vulnerable
}

total_labelled_records = sum(class_counts.values())

# Preserve the original class distribution.
desired_class_sizes = {
    1: round(
        SAMPLE_SIZE
        * class_counts[1]
        / total_labelled_records
    )
}

desired_class_sizes[0] = (
    SAMPLE_SIZE - desired_class_sizes[1]
)

print("Required sample sizes:", desired_class_sizes)

# Collect slightly more candidates so that duplicate removal
# still leaves enough records for each class.
candidate_sizes = {
    label: min(
        class_counts[label],
        math.ceil(required_size * 1.20) + 100
    )
    for label, required_size
    in desired_class_sizes.items()
}

print("Candidate reservoir sizes:", candidate_sizes)

rng = random.Random(RANDOM_SEED)

reservoirs = {
    0: [],
    1: [],
}

records_seen = {
    0: 0,
    1: 0,
}

required_columns = [
    "func",
    "target",
    "cwe",
    "project",
    "size",
]

for chunk in pd.read_csv(
    DATA_PATH,
    usecols=required_columns,
    chunksize=CHUNK_SIZE,
    encoding="utf-8",
):
    # Remove records that cannot be used for classification.
    chunk = chunk.dropna(
        subset=["func", "target"]
    )

    chunk = chunk[
        chunk["target"].isin([0, 1])
    ]

    for row in chunk.itertuples(index=False):
        label = int(row.target)
        records_seen[label] += 1

        record = {
            "func": row.func,
            "target": label,
            "cwe": row.cwe,
            "project": row.project,
            "size": row.size,
        }

        reservoir = reservoirs[label]
        reservoir_limit = candidate_sizes[label]

        if len(reservoir) < reservoir_limit:
            reservoir.append(record)
        else:
            replacement_index = rng.randrange(
                records_seen[label]
            )

            if replacement_index < reservoir_limit:
                reservoir[replacement_index] = record

print("Candidate sampling completed.")
print(
    "Candidate class 0:",
    len(reservoirs[0])
)
print(
    "Candidate class 1:",
    len(reservoirs[1])
)

Required sample sizes: {1: 591, 0: 9409}
Candidate reservoir sizes: {1: 810, 0: 11391}
Candidate sampling completed.
Candidate class 0: 11391
Candidate class 1: 810


In [10]:
# Clean candidate records, remove duplicates,
# create the final stratified dataset and save it locally.

candidate_df = pd.DataFrame(
    reservoirs[0] + reservoirs[1]
)

print("Initial candidate records:", len(candidate_df))

# Remove unusable records.
candidate_df = candidate_df.dropna(
    subset=["func", "target"]
).copy()

candidate_df["func"] = (
    candidate_df["func"]
    .astype(str)
    .str.strip()
)

candidate_df = candidate_df[
    candidate_df["func"].str.len() > 0
].copy()

# Identify source-code functions that appear with conflicting labels.
label_counts_per_function = (
    candidate_df
    .groupby("func")["target"]
    .nunique()
)

conflicting_functions = (
    label_counts_per_function[
        label_counts_per_function > 1
    ].index
)

print(
    "Functions with conflicting labels:",
    len(conflicting_functions)
)

# Remove ambiguous records.
candidate_df = candidate_df[
    ~candidate_df["func"].isin(conflicting_functions)
].copy()

records_before_deduplication = len(candidate_df)

# Keep only one copy of each source-code function.
candidate_df = (
    candidate_df
    .drop_duplicates(subset=["func"])
    .reset_index(drop=True)
)

records_after_deduplication = len(candidate_df)

print(
    "Records before duplicate removal:",
    records_before_deduplication
)

print(
    "Records after duplicate removal:",
    records_after_deduplication
)

# Select the required number of records from each class.
final_parts = []

for label, required_size in desired_class_sizes.items():
    class_candidates = candidate_df[
        candidate_df["target"] == label
    ]

    print(
        f"Unique candidates for class {label}:",
        len(class_candidates)
    )

    if len(class_candidates) < required_size:
        raise RuntimeError(
            f"Not enough unique records for class {label}. "
            f"Required: {required_size}, "
            f"available: {len(class_candidates)}"
        )

    sampled_class = class_candidates.sample(
        n=required_size,
        random_state=RANDOM_SEED
    )

    final_parts.append(sampled_class)

# Combine and shuffle both classes.
working_df = (
    pd.concat(final_parts, ignore_index=True)
    .sample(
        frac=1,
        random_state=RANDOM_SEED
    )
    .reset_index(drop=True)
)

print("\nFinal working dataset shape:")
print(working_df.shape)

print("\nFinal class distribution:")
print(
    working_df["target"]
    .value_counts()
    .sort_index()
)

print("\nFinal class percentages:")
print(
    working_df["target"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

# Save the processed dataset locally.
PROCESSED_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "diversevul_stratified_10000.csv"
)

PROCESSED_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)

working_df.to_csv(
    PROCESSED_PATH,
    index=False
)

print("\nProcessed dataset saved to:")
print(PROCESSED_PATH)

print(
    "Saved file exists:",
    PROCESSED_PATH.exists()
)

print(
    "Saved file size:",
    f"{PROCESSED_PATH.stat().st_size / (1024 * 1024):.2f} MB"
)

Initial candidate records: 12201
Functions with conflicting labels: 0
Records before duplicate removal: 12201
Records after duplicate removal: 12201
Unique candidates for class 1: 810
Unique candidates for class 0: 11391

Final working dataset shape:
(10000, 5)

Final class distribution:
target
0    9409
1     591
Name: count, dtype: int64

Final class percentages:
target
0    94.09
1     5.91
Name: proportion, dtype: float64

Processed dataset saved to:
f:\BSc (Hons) in Cyber Security\Projects\NLP_Group_02\data\processed\diversevul_stratified_10000.csv
Saved file exists: True
Saved file size: 12.69 MB
